# Bayesian hyperparameter optimisation (Optuna, TPE)

Run this **after** the model comparison, on the model you selected. It supports `random_forest`, `mlp` and `lightgbm`.

- Search: Optuna's TPE sampler (Tree-structured Parzen Estimator), which is Bayesian optimisation. Seeded for reproducibility.
- Objective: maximise the mean **group-CV observation-level Top-1** over the 5 shared trip-level folds of the 80 training trips. The student-test trips are never used inside the search.
- After the search, the best configuration is refitted on the same training rows as the untuned model and evaluated on the student-test trips. Tuned and untuned are then compared with identical metrics and data partitions.

Outputs go to `tuning/<model>/`: `trials.csv`, `best_params.json`, `tuned_vs_untuned.csv`, `model_tuned.joblib` and the plots.

In [ ]:
import os, sys, json, time
from pathlib import Path
# find the repo root (the folder that contains config.yaml) and make `pathsnap` importable
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the pathsnap-ml-hackathon repo")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathsnap.dataset import FEATURES
from pathsnap import modeling as M
print("repo root:", ROOT)

## Settings

In [ ]:
MODEL = "random_forest"        # "random_forest" | "mlp" | "lightgbm"
N_TRIALS = 30
ROWS_PER_FOLD = 40_000          # fit rows per training fold during the search (validation folds stay full)
FINAL_ROWS_PER_FOLD = None      # None = same subsample as the untuned run in models/<MODEL>/metrics.json
OUT = ROOT / "tuning" / MODEL
OUT.mkdir(parents=True, exist_ok=True)

untuned_path = ROOT / "models" / MODEL / "metrics.json"
untuned = json.loads(untuned_path.read_text()) if untuned_path.exists() else None
if FINAL_ROWS_PER_FOLD is None and untuned:
    FINAL_ROWS_PER_FOLD = untuned.get("train_rows_per_fold")
print("untuned run found:", untuned is not None, "| final rows/fold:", FINAL_ROWS_PER_FOLD)

## Search space

In [ ]:
import optuna
from sklearn.ensemble import RandomForestClassifier

def build(params):
    if MODEL == "random_forest":
        m = RandomForestClassifier(n_jobs=-1, random_state=M.SEED, **params)
        fit = lambda mod, df: mod.fit(df[FEATURES], df["label"], sample_weight=df["trip_weight"].values)
    elif MODEL == "mlp":
        p = dict(params); p["hidden_layer_sizes"] = tuple(int(x) for x in p.pop("layers").split("-"))
        m = M.EpochMLP(max_epochs=40, patience=4, seed=M.SEED, **p)
        fit = lambda mod, df: mod.fit_df(df)
    elif MODEL == "lightgbm":
        import lightgbm as lgb
        m = lgb.LGBMClassifier(random_state=M.SEED, verbose=-1, subsample_freq=1, **params)
        fit = lambda mod, df: mod.fit(df[FEATURES], df["label"], sample_weight=df["trip_weight"].values)
    return m, fit

def suggest(trial):
    if MODEL == "random_forest":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
            max_depth=trial.suggest_int("max_depth", 6, 30),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 50, log=True),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5]),
        )
    if MODEL == "mlp":
        return dict(
            layers=trial.suggest_categorical("layers", ["32", "64", "128", "64-32", "128-64", "128-64-32"]),
            alpha=trial.suggest_float("alpha", 1e-6, 1e-2, log=True),
            learning_rate_init=trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
            batch_size=trial.suggest_categorical("batch_size", [256, 512, 1024]),
        )
    if MODEL == "lightgbm":
        return dict(
            n_estimators=trial.suggest_int("n_estimators", 100, 800, step=50),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 200, log=True),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        )

## Objective: mean group-CV Top-1 (with median pruning after each fold)

In [ ]:
from pathsnap.evaluate import observation_metrics
tr_all, _, tr_pts, _ = M.load_data()
print(f"fit rows per training fold during search: {ROWS_PER_FOLD}; validation folds are full")

def objective(trial):
    params = suggest(trial)
    scores = []
    for k, f in enumerate(sorted(tr_all["fold"].unique())):
        a = M.subsample_by_fold(tr_all[tr_all.fold != f], ROWS_PER_FOLD)
        v = tr_all[tr_all.fold == f]
        m, fit = build(params)
        m = fit(m, a)
        r = observation_metrics(v, m.predict_proba(v[FEATURES])[:, 1], tr_pts[tr_pts.fold == f])
        scores.append(r["top1_accuracy"])
        trial.set_user_attr(f"fold{f}_log_loss", r["log_loss"])
        trial.report(float(np.mean(scores)), k)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("cv_top1_std", float(np.std(scores)))
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize", study_name=f"pathsnap_{MODEL}",
                            sampler=optuna.samplers.TPESampler(seed=M.SEED),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
t0 = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
search_seconds = time.perf_counter() - t0
print(f"search time: {search_seconds/60:.1f} min   best CV Top-1: {study.best_value:.4f}")
print("best params:", study.best_params)

In [ ]:
trials = study.trials_dataframe()
trials.to_csv(OUT / "trials.csv", index=False)
(OUT / "best_params.json").write_text(json.dumps({"model": MODEL, "best_params": study.best_params,
    "best_cv_top1": study.best_value, "n_trials": len(study.trials),
    "n_complete": sum(t.state.name == "COMPLETE" for t in study.trials),
    "n_pruned": sum(t.state.name == "PRUNED" for t in study.trials),
    "search_rows_per_fold": ROWS_PER_FOLD, "search_seconds": search_seconds,
    "sampler": "TPE (seed 42)", "pruner": "MedianPruner"}, indent=2, default=str))
trials[["number", "value", "state"] + [c for c in trials.columns if c.startswith("params_")]].sort_values("value", ascending=False).head(10)

## Optimisation history and parameter importance

In [ ]:
done = trials[trials.state == "COMPLETE"].sort_values("number")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(done["number"], done["value"], "o", alpha=0.6, label="trial")
ax[0].plot(done["number"], done["value"].cummax(), "r-", label="best so far")
ax[0].set(xlabel="trial", ylabel="mean CV Top-1", title=f"Optuna TPE history ({MODEL})"); ax[0].legend()
try:
    imp = pd.Series(optuna.importance.get_param_importances(study)).sort_values()
    imp.plot.barh(ax=ax[1], color="#6a1b9a"); ax[1].set_title("Hyperparameter importance (fANOVA)")
except Exception as e:
    ax[1].text(0.1, 0.5, f"importance unavailable: {e}")
plt.tight_layout(); plt.savefig(OUT / "optimisation_history.png", dpi=130); plt.show()

## Refit tuned model and compare with the untuned model (same data, same metrics)

In [ ]:
tr, te, tr_pts, te_pts = M.load_data()
params = dict(study.best_params)
make_tuned = lambda: build(params)[0]
fit_tuned = build(params)[1]
cv_t = M.cross_validate(make_tuned, fit_tuned, tr, tr_pts, FINAL_ROWS_PER_FOLD)
model_t, scores_t, test_t, train_s_t, infer_ms_t = M.fit_and_test(make_tuned, fit_tuned, tr, te, te_pts, FINAL_ROWS_PER_FOLD)
res_t = M.save_run(OUT, f"{MODEL}_tuned", untuned.get("algorithm_family") if untuned else MODEL, params,
                   cv_t, test_t, train_s_t, infer_ms_t, model=None, rows_per_fold=FINAL_ROWS_PER_FOLD)
import joblib
joblib.dump(model_t, OUT / "model_tuned.joblib", compress=3)
print("tuned model saved:", (OUT / "model_tuned.joblib").stat().st_size / 1e6, "MB")

In [ ]:
def row(r, label):
    t = r["student_test"]
    return {"model": label, "CV macro F1": r["cv_f1_macro_mean"], "CV Top-1": r["cv_top1_mean"],
            "Student-test Top-1": t["top1_accuracy"], "Student-test macro F1": t["f1_macro"],
            "Log loss": t["log_loss"], "Inference (ms / 1k rows)": r["inference_ms_per_1000_rows"],
            "Train time (s)": r["train_seconds"]}
comp = pd.DataFrame(([row(untuned, "untuned")] if untuned else []) + [row(res_t, "tuned (Bayesian)")])
comp.to_csv(OUT / "tuned_vs_untuned.csv", index=False)
comp.round(4)

In [ ]:
if untuned:
    cols = ["CV macro F1", "CV Top-1", "Student-test Top-1", "Student-test macro F1"]
    x = np.arange(len(cols))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].bar(x - 0.2, comp.iloc[0][cols], 0.4, label="untuned")
    ax[0].bar(x + 0.2, comp.iloc[1][cols], 0.4, label="tuned")
    ax[0].set_xticks(x, cols, rotation=15); ax[0].set_ylim(0.5, 1); ax[0].legend()
    ax[0].set_title(f"{MODEL}: untuned vs tuned")
    ax[1].bar(["untuned", "tuned"], comp["Log loss"], color=["#90a4ae", "#c62828"])
    ax[1].set_title("Student-test log loss")
    plt.tight_layout(); plt.savefig(OUT / "tuned_vs_untuned.png", dpi=130); plt.show()

## Notes for the report (answer from YOUR results)
1. Which hyperparameters were searched: see `suggest()`.
2. Ranges and distributions: see `suggest()`; the `log=True` ones are log-uniform.
3. Objective: mean 5-fold trip-level CV Top-1 on the training trips.
4. Number of trials: `N_TRIALS`, with the completed and pruned counts in `best_params.json`.
5. Best values: `best_params.json`.
6. Did tuning help? Read `tuned_vs_untuned.csv`. If not, say so honestly and explain why.